<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/03_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##Import libraries
import os
import joblib
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from google.colab import drive
##Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
##Metrics and Processing
from sklearn.metrics import classification_report, hamming_loss, accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import hamming_loss, f1_score, precision_score, recall_score
from scipy.sparse import hstack, csr_matrix

In [ ]:
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/.shortcut-targets-by-id/1SLhiH7VulyiPZ7L846kJBEbN1pa4c4zU/Multi_Label_Email_Risk_Detection/"
DATA_RAW_DIR = f"{PROJECT_DIR}/data/raw"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"
REPORTS_DIR = f"{PROJECT_DIR}/reports"
FEATURE_DIR = f"{PROJECT_DIR}/data/processed/features"
os.chdir(PROJECT_DIR)

print("Current folder:", os.getcwd())
print("Processed files:", os.listdir(DATA_PROCESSED_DIR))

ValueError: mount failed

In [ ]:
combined_df = pd.read_csv(f"{DATA_PROCESSED_DIR}/combined_dataset.csv")

risk_cols = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

print("Dataset shape:", combined_df.shape)
display(combined_df.head())

In [ ]:
#FEATURE_DIR = f"{PROJECT_DIR}/data/processed/features"

print(os.listdir(FEATURE_DIR))

In [ ]:
# ------------------------------------------------------------
# Load Final Feature Engineering Outputs
# ------------------------------------------------------------




# Load feature matrices created in feature engineering notebook

X_train = joblib.load(
    f"{FEATURE_DIR}/X_train_features.pkl"
)

X_test = joblib.load(
    f"{FEATURE_DIR}/X_test_features.pkl"
)


# Load multi-label targets

Y_train = joblib.load(
    f"{FEATURE_DIR}/y_train.pkl"
)

Y_test = joblib.load(
    f"{FEATURE_DIR}/y_test.pkl"
)


# Dictionary for storing model results

results = {}


# Verify dimensions

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("Training labels:", Y_train.shape)
print("Testing labels:", Y_test.shape)


In [ ]:
##Start Tools
#vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

##Get Label Encoded on Category column
#Y = combined_df[risk_cols]

##Split Dataframe with source stratified to avoid bnias towards one type
#train_df, test_df = train_test_split(combined_df, test_size=0.2, random_state=26, stratify=combined_df['source'])

###Fit vectorizer only on training data
#X_train = vectorizer.fit_transform(train_df['text'])
#X_test = vectorizer.transform(test_df['text'])

##Encode targets
#Y_train = train_df[risk_cols]
#Y_test = test_df[risk_cols]

##Empty List for Results
#results = {}

####Logistic Model
$$z_k =w_k^T x + b_k$$

$$ P(y_k = 1| x) = σ(z_k) = \frac{1}{1+e^{-zk}}$$

In [ ]:
##Logistic Model
from sklearn.multioutput import MultiOutputClassifier

log_model = MultiOutputClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))

##Train Model
log_model.fit(X_train, Y_train)

##Make Predications
log_preds = log_model.predict(X_test)
log_probs = log_model.predict_proba(X_test)
##Get Metrics
acc = accuracy_score(Y_test, log_preds)
prec = precision_score(Y_test, log_preds, average='weighted', zero_division=0)
rec = recall_score(Y_test, log_preds, average='weighted', zero_division=0)
f1 = f1_score(Y_test, log_preds, average='weighted', zero_division=0)
hl = hamming_loss(Y_test, log_preds)

##Store Metrics For comparision later
results['LogisticRegression'] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,'hamming_loss': hl }
##Print here for Claritry/Checks
print(f"Metrics for Logistic Regression:")
print(f"Accuracy: {acc}, Precision: {prec}, Recall: {rec}, F1: {f1},Hamming Loss: {hl}")

#Save the model
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(
    log_model,
    f"{MODELS_DIR}/logistic_regression_model.pkl"
)

# Save the predictions
joblib.dump(
    log_preds,
    f"{REPORTS_DIR}/logistic_regression_predictions.pkl"
)


##Random Forest
Decision Tree Node (Gini Index)-
$$G = 1 - \sum_{k=1}^{K}\hat p_{mk}(1-\hat p_{mk})$$

Forest Level Probability Across Trees-

$$\hat{p_k} = \frac{1}{N}\sum_{n=1}^{N}T_n(x)_k$$

In [ ]:
##Random Forest Model
RFor_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, max_depth=10, random_state=26))

##Train Model
RFor_model.fit(X_train, Y_train)

##Make predictions
RFor_preds = RFor_model.predict(X_test)

##Get Metrics
acc = accuracy_score(Y_test, RFor_preds)
prec = precision_score(Y_test, RFor_preds, average='weighted', zero_division=0)
rec = recall_score(Y_test, RFor_preds, average='weighted', zero_division=0)
f1 = f1_score(Y_test, RFor_preds, average='weighted', zero_division=0)
hl = hamming_loss(Y_test, RFor_preds)

##Store Metrics
results['RandomForest'] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,'hamming_loss': hl}

print(f"Metrics for Random Forest:")
print(f"Accuracy: {acc}, Precision: {prec}, Recall: {rec}, F1: {f1},Hamming Loss: {hl}")

# Save the model
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(
    RFor_model,
    f"{MODELS_DIR}/random_forest_model.pkl"
)

# Save the predictions
joblib.dump(
    RFor_preds,
    f"{REPORTS_DIR}/random_forest_predictions.pkl"
)


####Gradient Boosting (XGBoost)
$$\hat{f}(x) = \sum_{t=1}^{D}λ \hat f^t(x)$$

$\hat f^t$ - Decsion Tree  @ t

$x$ - Input

#####Minimize Loss Function
$$ \text{Loss}^{(t)} = \sum_{i=1}^{n} L(y_i, \hat{y}_i^{(t-1)} + λ \hat f^t(x_i)) + \Omega(\hat f^t)$$

$ L $ - Log-loss

$ y_i $ - Ground Truth

$ \hat{y}_i^{(t-1)} $ - Current Prediction

$λ$ - Learning Rate

$\hat f^t(x_i)$ - New Tree

$ \Omega(\hat f^t) $ -Regularization term (Prevent Overfitting)


In [ ]:
##Gradient Boosting (XGBoost)
X_model = MultiOutputClassifier(XGBClassifier(
  n_estimators=100,  ##100 Trees
  learning_rate=0.1,
  max_depth=10,  ##Regularization set at 10
  tree_method='hist',
  n_jobs=-1,
  eval_metric='logloss'))
##Train Model
X_model.fit(X_train, Y_train, verbose=True)
##Make Predictions
X_preds = X_model.predict(X_test)
##Get Metrics
acc = accuracy_score(Y_test, X_preds)
prec = precision_score(Y_test, X_preds, average='weighted', zero_division=0)
rec = recall_score(Y_test, X_preds, average='weighted', zero_division=0)
f1 = f1_score(Y_test, X_preds, average='weighted', zero_division=0)
hl = float(hamming_loss(Y_test, X_preds))
micro_f1 = f1_score(Y_test, X_preds, average='micro')
macro_f1 = f1_score(Y_test, X_preds, average='macro')

##Store Metrics
results['XGBoost'] = {'subset_accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,'hamming_loss': hl, 'micro_f1':micro_f1, 'macro_f1':macro_f1 }

print(f"Metrics for XGBoost:")
print(f"Accuracy: {acc}, Precision: {prec}, Recall: {rec}, F1: {f1}, Hamming Loss: {hl}, 'micro_f1': {micro_f1}, 'macro_f1': {macro_f1}")

# Save the model
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(
    X_model,
    f"{MODELS_DIR}/xgboost_model.pkl"
)

# Save the predictions
joblib.dump(
    X_preds,
    f"{REPORTS_DIR}/xgboost_predictions.pkl"
)


####Support Vector Machines
$$ f_k(x) = w_k^T x+b_k $$

$ w_k^T $ - Weight Vector for Class k

$ b_k $ Bias for class k

In [ ]:
from sklearn.svm import LinearSVC

##Support Vector Machines
SVM_model = MultiOutputClassifier(make_pipeline(
    StandardScaler(with_mean=False),
    LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)
))

##Train and Predict
SVM_model.fit(X_train, Y_train)
SVM_preds = SVM_model.predict(X_test)

##Overall Evaluation Metrics
acc = accuracy_score(Y_test, SVM_preds)
prec = precision_score(Y_test, SVM_preds, average='weighted', zero_division=0)
rec = recall_score(Y_test, SVM_preds, average='weighted', zero_division=0)
f1 = f1_score(Y_test, SVM_preds, average='weighted', zero_division=0)
hl = hamming_loss(Y_test, SVM_preds)
micro_f1 = f1_score(Y_test, SVM_preds, average='micro')
macro_f1 = f1_score(Y_test, SVM_preds, average='macro')

##Print Metrics for Each Risk Category
for i, col in enumerate(risk_cols):
    print(f"\n{col}")
    print(classification_report(Y_test.iloc[:, i], SVM_preds[:, i], zero_division=0))

##Store and Display Results
results["SVM"] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, 'hamming_loss': hl,"micro_f1": micro_f1, "macro_f1": macro_f1}

print("\nMetrics for SVM:")
print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1 Score: {f1:.4f}, Hamming Loss: {hl:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")
##Note: Some risk categories may be easier because they contain strong indicators like credentials, customer records, or template-based language.

# Save the model
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(
    SVM_model,
    f"{MODELS_DIR}/svm_model.pkl"
)

# Save the predictions
joblib.dump(
    SVM_preds,
    f"{REPORTS_DIR}/svm_predictions.pkl"
)


####Multi-Layer Perception Model
Activation (Sigmoid)-
$$ σ(z_k) = \frac{1}{1 + e^{-z_k}}$$



Loss Binary Cross Entropy)-
$$Loss = -\sum_{k=1}^{K}[y_k\text{log}(p_k) +(1-y_k)\text{log}(1-p_k)]
$$
Where yk is the ground truth

In [ ]:
# ------------------------------------------------------------
# Convert Sparse Matrix to Dense for Neural Network
# ------------------------------------------------------------
# TensorFlow/Keras Dense layers require dense input.
# Other models (SVM, Logistic Regression) can use sparse matrices.

X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

print("Dense training shape:", X_train_dense.shape)
print("Dense testing shape:", X_test_dense.shape)

##Get Input and Output Dimensions
input_shape = X_train_dense.shape[1]
output_shape = len(risk_cols)

##Define Multi-Layer Perceptron Model
MLP_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(input_shape,)),
    layers.Dropout(0.3),  ##Prevent Overfitting
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(output_shape, activation='sigmoid')  ##One probability per risk category
])

##Compile Model
MLP_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
   metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

##Train Model
history = MLP_model.fit(
    X_train_dense,
    Y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

##Predict Risk Probabilities
mlp_probs = MLP_model.predict(X_test_dense)

##Convert Probabilities to Binary Predictions
mlp_preds = (mlp_probs >= 0.4).astype(int)

##Overall Evaluation Metrics
acc = accuracy_score(Y_test, mlp_preds)
prec = precision_score(Y_test, mlp_preds, average='weighted', zero_division=0)
rec = recall_score(Y_test, mlp_preds, average='weighted', zero_division=0)
f1 = f1_score(Y_test, mlp_preds, average='weighted', zero_division=0)
hl = hamming_loss(Y_test, mlp_preds)
micro_f1 = f1_score(Y_test, mlp_preds, average='micro')
macro_f1 = f1_score(Y_test, mlp_preds, average='macro')

##Print Metrics for Each Risk Category
for i, col in enumerate(risk_cols):
    print(f"\n{col}")
    print(classification_report(
            Y_test.iloc[:, i],
            mlp_preds[:, i],
            zero_division=0))

##Store Overall Results
results["NeuralNetwork"] = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    'hamming_loss': hl,
    "micro_f1": micro_f1,
    "macro_f1": macro_f1
    }

##Display Overall Metrics
print("\nMetrics for Neural Network:")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Hamming Loss: {hl:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}")


# Save the model
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(
    MLP_model,
    f"{MODELS_DIR}/mlp_model.pkl"
)

# Save the predictions
joblib.dump(
    mlp_preds,
    f"{REPORTS_DIR}/mlp_predictions.pkl"
)